<a href="https://colab.research.google.com/github/rania10082004/Syntecxhub_-Movie_-Recommendation_System/blob/main/MOVIE_RECOMMENDATION_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
parasharmanas_movie_recommendation_system_path = kagglehub.dataset_download('parasharmanas/movie-recommendation-system')
ibtesama_getting_started_with_a_movie_recommendation_system_path = kagglehub.notebook_output_download('ibtesama/getting-started-with-a-movie-recommendation-system')

print('Data source import complete.')


In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
from sklearn.feature_extraction.text import CountVectorizer

# Similarity Calculation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Ignore Warnings
import warnings
warnings.filterwarnings('ignore')

| Library           | Purpose                           |
| ----------------- | --------------------------------- |
| pandas            | Data loading and manipulation     |
| numpy             | Numerical calculations            |
| matplotlib        | Basic plotting and charts         |
| seaborn           | Advanced data visualization       |
| CountVectorizer   | Convert text data into vectors    |
| cosine_similarity | Measure similarity between movies |
| train_test_split  | Split data for training/testing   |
| WordCloud         | Generate movie title word clouds  |
| warnings          | Hide unnecessary warnings         |


In [ ]:
import shutil
import os

# Use the path provided by kagglehub for the dataset
shutil.copy(os.path.join(parasharmanas_movie_recommendation_system_path, "movies.csv"), "movies.csv")
shutil.copy(os.path.join(parasharmanas_movie_recommendation_system_path, "ratings.csv"), "ratings.csv")

In [ ]:
df_movies = pd.read_csv("movies.csv")
df_rating = pd.read_csv("ratings.csv")

In [ ]:
df_movies.head()

In [ ]:
df_rating.head()

In [ ]:
print("Movies Dataset Shape :", df_movies.shape)

In [ ]:
print("Ratings Dataset Shape :", df_rating.shape)

In [ ]:
df_movies.info()

In [ ]:
df_rating.info()

In [ ]:
df_movies.describe().T

In [ ]:
df_rating.describe().T

In [ ]:
# Missing values in Movies Dataset
df_movies.isnull().sum()

In [ ]:
# Missing values in Ratings Dataset
df_rating.isnull().sum()

In [ ]:
# Duplicate values in Movies Dataset
df_movies.duplicated().sum()

In [ ]:
# Duplicate values in Ratings Dataset
df_rating.duplicated().sum()

In [ ]:
# Fill missing genres with empty string
df_movies['genres'] = df_movies['genres'].fillna('')

In [ ]:
# Merge datasets
df = pd.merge(df_rating, df_movies, on='movieId')

# Display merged dataset
df.head()

In [ ]:
print("Merged Dataset Shape :", df.shape)

In [ ]:
df.columns

In [ ]:
# Average rating of each movie
top_rated = df.groupby('title')['rating'].mean().sort_values(ascending=False)

# Convert into dataframe
top_rated = pd.DataFrame(top_rated)

# Display top 10 rated movies
top_rated.head(10)

In [ ]:
# Count ratings for each movie
popular_movies = df.groupby('title')['rating'].count().sort_values(ascending=False)

# Convert into dataframe
popular_movies = pd.DataFrame(popular_movies)

# Display top 10 popular movies
popular_movies.head(10)

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df['rating'])

plt.xlabel("Ratings")
plt.ylabel("Number of Ratings")
plt.title("Distribution of Movie Ratings")

plt.show()

In [ ]:
# Split genres
genres = df_movies['genres'].str.split('|').explode()

# Count genres
genre_count = genres.value_counts()

# Display genre counts
genre_count

In [ ]:
plt.figure(figsize=(10,6))

genre_count.plot(kind='bar')

plt.xlabel("Genres")
plt.ylabel("Count")
plt.title("Movie Genre Distribution")

plt.show()

In [ ]:
# Extract year
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')

# Display first rows
df_movies.head()

In [ ]:
# Extract year
df_movies['year'] = df_movies['title'].str.extract(r'\((\d{4})\)')

# Display first rows
df_movies.head()

In [ ]:

movies_per_year = df_movies['year'].value_counts().sort_index()

plt.figure(figsize=(12,5))

movies_per_year.plot()

plt.xlabel("Year")
plt.ylabel("Number of Movies")
plt.title("Movies Released Per Year")

plt.show()


In [ ]:
movies_data = df_movies[['movieId', 'title', 'genres']]

movies_data.head()

In [ ]:
movies_data['genres'] = movies_data['genres'].fillna('')

In [ ]:
# Initialize CountVectorizer
cv = CountVectorizer(tokenizer=lambda x: x.split('|'))

# Convert genres into matrix
count_matrix = cv.fit_transform(movies_data['genres'])

print(count_matrix)

In [ ]:
print(count_matrix.shape)

In [ ]:
tfidf = TfidfVectorizer(token_pattern=r'[^|]+')

tfidf_matrix = tfidf.fit_transform(df_movies['genres'])


In [ ]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

model.fit(tfidf_matrix)

In [ ]:
def recommend(movie_name):

    movie_name = movie_name.lower()

    # Find movie index
    idx = df_movies[
        df_movies['title'].str.lower() == movie_name
    ].index

    if len(idx) == 0:
        print("Movie not found!")
        return

    idx = idx[0]

    # Find nearest neighbors
    distances, indices = model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=11
    )

    print(f"\n🎬 Recommended Movies for '{df_movies.iloc[idx]['title']}'\n")

    for i in range(1, len(indices[0])):

        movie_idx = indices[0][i]

        print(df_movies.iloc[movie_idx]['title'])

In [ ]:
recommend("Toy Story (1995)")

In [ ]:
recommend("Jumanji (1995)")

In [ ]:
recommend("Heat (1995)")

In [ ]:
recommend("ABC Movie")

In [ ]:
# ==========================================
# USER-MOVIE RATINGS HEATMAP
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reload data after kernel restart
df_rating = pd.read_csv("ratings.csv", dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'})

# Filter to top 30 users and top 30 movies BEFORE pivot
top_movies = df_rating['movieId'].value_counts().head(30).index
top_users  = df_rating['userId'].value_counts().head(30).index

filtered = df_rating[
    df_rating['movieId'].isin(top_movies) &
    df_rating['userId'].isin(top_users)
]

# Create User-Movie Matrix
movie_matrix = filtered.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

# Select smaller portion for visualization
heatmap_data = movie_matrix.iloc[:30, :30]

# Create Heatmap
plt.figure(figsize=(15,10))
sns.heatmap(
    heatmap_data,
    cmap='coolwarm',
    linewidths=0.5,
    linecolor='white',
    cbar=True
)

# Titles and Labels
plt.title("User-Movie Ratings Heatmap", fontsize=20)
plt.xlabel("Movie IDs", fontsize=14)
plt.ylabel("User IDs", fontsize=14)

plt.show()

In [ ]:
import gradio as gr

def get_recommendations(movie_title):
    if not movie_title.strip():
        return "### ⚠️ Please enter a movie name."

    movie_name_lower = movie_title.lower()
    idx_list = df_movies[df_movies['title'].str.lower().str.contains(movie_name_lower)].index

    if len(idx_list) == 0:
        return "### ❌ Movie not found!"

    idx = idx_list[0]
    original_title = df_movies.iloc[idx]['title']
    distances, indices = model.kneighbors(tfidf_matrix[idx], n_neighbors=11)

    # High contrast results for clear visibility
    results = f"## 🎬 Recommendations for: <span style='color:#FFD700'>{original_title}</span>\n\n"
    for i in range(1, len(indices[0])):
        movie_idx = indices[0][i]
        results += f"⭐ <span style='color:white'>{df_movies.iloc[movie_idx]['title']}</span>\n\n"

    return results

# Forceful High-Contrast CSS
custom_css = """
    body, .gradio-container { background-color: #0b0f19 !important; color: white !important; }
    #title-text h1 { color: #FFD700 !important; text-shadow: 2px 2px 4px #000; }
    .label-text, p, span, h3 { color: #ffffff !important; font-weight: bold !important; }
    input, textarea { background-color: #1f2937 !important; color: white !important; border: 1px solid #4b5563 !important; }
    #recommend-btn {
        background: linear-gradient(90deg, #1e40af, #3b82f6) !important;
        color: white !important;
        border: none !important;
        font-size: 18px !important;
    }
    .markdown-text { color: white !important; line-height: 1.6; }
"""

with gr.Blocks(theme=gr.themes.Monochrome(), css=custom_css) as demo:
    gr.Markdown("# 🎬 SyntecxHub Movie Recommender", elem_id="title-text")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🔍 SEARCH MOVIE")
            movie_input = gr.Textbox(
                label="Enter Title",
                placeholder="e.g. Toy Story (1995)",
                lines=1
            )
            recommend_btn = gr.Button("GET RECOMMENDATIONS", elem_id="recommend-btn")

            gr.Examples(
                examples=["Toy Story", "Jumanji", "Heat"],
                inputs=movie_input
            )

        with gr.Column(scale=1):
            gr.Markdown("### 📋 TOP SUGGESTIONS")
            output_display = gr.Markdown(elem_classes="markdown-text")

    recommend_btn.click(fn=get_recommendations, inputs=movie_input, outputs=output_display)

demo.launch(debug=True)